In [1]:
# Cài nnU-Net bản chuẩn từ PyPI (Nhanh và sạch)
!pip install -q nnunetv2

# Cài thêm các thư viện bổ trợ nếu thiếu
!pip install -q nibabel matplotlib medpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 7.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!nvidia-smi

Fri Mar 20 11:17:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# A. Set up

## 1. Giải nén `nnUNet_raw` & `nnUNet_preprocessed`

In [3]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [4]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [02:03<00:00, 15.00it/s]


In [5]:
unzip(PRE_ZIP, PRE_DIR)

100%|██████████| 2590/2590 [04:20<00:00,  9.96it/s]


## 2. Set biến môi trường nnU-Net v2

In [6]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results


## 3. Check split file

### 3.1 Validation set

In [7]:
!cp /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json \
/content/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json

In [8]:
import json

split_path = "/content/nnUNet_preprocessed/Dataset101_BraTS2020/splits_final.json"

with open(split_path) as f:
    splits = json.load(f)

print(f"Số folds: {len(splits)}")
print("Fold 0:", splits[0].keys())
print("Val size fold 0:", len(splits[0]["val"]))

Số folds: 5
Fold 0: dict_keys(['train', 'val'])
Val size fold 0: 59


### 3.2 Test set

In [9]:
import json

split_path = "/content/drive/MyDrive/NCKH/nnUnet/data/experiments/fixed_test.json"

with open(split_path) as f:
    splits = json.load(f)

print(f"Số folds: {len(splits)}")

for fold_name, cases in splits.items():
    print(f"{fold_name}: {len(cases)} cases")

Số folds: 5
fold_0: 74 cases
fold_1: 74 cases
fold_2: 74 cases
fold_3: 74 cases
fold_4: 73 cases


# B. Inference Uncertainty Map

In [ ]:
# CHẾ ĐỘ 1: CHẠY MODEL EDL (UNCERTAINTY DECOMPOSITION)
# ------------------------------------------------------------------
# Script này sẽ thực hiện quy trình full:
# 1. Tự động "tiêm" (inject) file EDLTrainer.py vào thư viện nnU-Net hệ thống.
# 2. Load trọng số Model EDL (checkpoint_best.pth) từ đường dẫn config 'edl'.
# 3. Chạy Inference và tính toán toán học để tách Uncertainty thành:
#    - Aleatoric (Nhiễu dữ liệu)
#    - Epistemic (Mô hình chưa biết)
# 4. Lưu kết quả Segmentation + 3 bản đồ Uncertainty (.nii.gz).

In [12]:
# Đổi tên file cho đúng chuẩn (giải nén còn thừa .gz)
!for f in /content/nnUNet_raw/Dataset101_BraTS2020/imagesTr/*.nii.gz; do mv "$f" "${f%.gz}"; done

## 1. Validation Set

### Fold 0

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 0 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 0 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 0: 59 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 59 cases (Filtered).
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/val

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_006...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_006.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_006.nii -> KHÔNG
100% 8/8 [00:04<00:00,  1.63it/s]
1

### Fold 1

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 1 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 1 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 1: 59 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 59 cases (Filtered).
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/val

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_013...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_013.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_013.nii -> KHÔNG
100% 8/8 [00:04<00:00,  1.61it/s]
1

### Fold 2

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 2 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 2 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 2: 59 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 59 cases (Filtered).
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/val

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_001...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii -> KHÔNG
100% 8/8 [00:04<00:00,  1.62it/s]
1

### Fold 3

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 3 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 3 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 3: 59 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 59 cases (Filtered).
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/val

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_005...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_005.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_005.nii -> KHÔNG
100% 8/8 [00:04<00:00,  1.61it/s]
1

### Fold 4

In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 4 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 4 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 4: 59 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 59 cases (Filtered).
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/val

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_002...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii -> KHÔNG
100% 8/8 [00:04<00:00,  1.61it/s]
1

## 2. Test set - fold test 0

In [13]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_250_fixed_split --fold 2 --test_fold 0 --run_mode fixed_test

🏁 --- STARTING PIPELINE | MODE: EDL_250_FIXED_SPLIT | FOLD: 2 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
📂 Đã load danh sách Test (fold_0): 74 ca.
⚙️ Mode: FIXED TEST | Model (Weights): Fold 2 | Data: Test Fold 0 -> 74 cases.
📁 Dữ liệu sẽ được lưu tại: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres/test

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_011...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii -> KHÔNG
100% 8/8 [00:05

##